In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error

# LOAD DATA
data = pd.read_csv("Delhi_v2.csv")

# Cleaning
data = data.drop(columns=["Unnamed: 0", "Price_sqft", "desc", "Landmarks", "Lift"], errors="ignore")

data["parking"] = data["parking"].fillna(0)
data["Balcony"] = data["Balcony"].fillna(0)

data["Status"] = data["Status"].fillna(data["Status"].mode()[0])
data["Furnished_status"] = data["Furnished_status"].fillna("unknown")

# clustering
geo = data[["latitude", "longitude"]]

scaler = StandardScaler()
geo_scaled = scaler.fit_transform(geo)

kmeans = KMeans(n_clusters=4, random_state=42)
data["location_cluster"] = kmeans.fit_predict(geo_scaled)

data = data.drop(columns=["latitude", "longitude"])

# Target
data["log_price"] = np.log(data["price"])

y = data["log_price"]
X = data.drop(columns=["log_price", "price"])

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Pipeline
num_attributes = ["area", "Bedrooms", "Bathrooms", "Balcony", "parking"]

categorical_cols = [
    "Status",
    "Furnished_status",
    "type_of_building",
    "neworold",
    "location_cluster"
]

num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_attributes),
    ("cat", cat_pipeline, categorical_cols)
])

model_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

# Train
model_pipeline.fit(X_train, y_train)

# Evaluation
y_pred_log = model_pipeline.predict(X_test)

y_pred_actual = np.exp(y_pred_log)
y_test_actual = np.exp(y_test)

mae = mean_absolute_error(y_test_actual, y_pred_actual)

print("MAE:", mae)

# Save
joblib.dump(model_pipeline, "model.pkl")
joblib.dump(mae, "mae.pkl")

print("Model + MAE saved successfully.")

MAE: 1571781.2585238505
Model + MAE saved successfully.
